# Gold_scrapper_run_quality

## Import and load data

In [40]:
import sqlite3

import pandas as pd

from ecommerce_ingestion.config.constants import DB_OUTPUT_BRONZE, DB_OUTPUT_GOLD


In [41]:

conn = sqlite3.connect(DB_OUTPUT_BRONZE / "oxylabs_sandbox_scraper.db")
gold_scrapper_run_quality = pd.read_sql("SELECT * FROM runs", conn)
gold_scrapper_run_quality.head()

,id,run_type,source_site,status,started_at,finished_at,error_message
0,8951ba1e-7120-4e31-b8c8-7d0012d0de8d,catalog,oxylabs_sandbox,succeeded,2026-05-14T23:56:10.630817,2026-05-14T23:57:58.897352,None
1,15e5f9d8-8482-4966-8c0d-3de4cc34f7de,catalog,oxylabs_sandbox,succeeded,2026-05-16T16:52:42.249897,2026-05-16T16:54:40.928026,None


In [42]:
gold_scrapper_run_quality.shape

(2, 7)

## Normalize and Fill NaNs

In [43]:
gold_scrapper_run_quality.isna().sum()

id               0
run_type         0
source_site      0
status           0
started_at       0
finished_at      0
error_message    2
dtype: int64

In [44]:
gold_scrapper_run_quality[
    "error_message"] =gold_scrapper_run_quality["error_message"].fillna("No error")

In [45]:
gold_scrapper_run_quality["started_at"] = pd.to_datetime(gold_scrapper_run_quality[
    "started_at"])
gold_scrapper_run_quality["finished_at"] = pd.to_datetime(gold_scrapper_run_quality[
    "finished_at"])

gold_scrapper_run_quality["duration"] = (gold_scrapper_run_quality[
    "finished_at"] - gold_scrapper_run_quality[
        "started_at"]).dt.total_seconds().astype(int)

In [46]:
gold_scrapper_run_quality.head()

,id,run_type,source_site,status,started_at,finished_at,error_message,duration
0,8951ba1e-7120-4e31-b8c8-7d0012d0de8d,catalog,oxylabs_sandbox,succeeded,2026-05-14 23:56:10.630817,2026-05-14 23:57:58.897352,No error,108
1,15e5f9d8-8482-4966-8c0d-3de4cc34f7de,catalog,oxylabs_sandbox,succeeded,2026-05-16 16:52:42.249897,2026-05-16 16:54:40.928026,No error,118


## Save data

In [47]:
gold_scrapper_run_quality.to_parquet(DB_OUTPUT_GOLD / 
                                     "gold_scrapper_run_quality.parquet", index=False)